# Lab_07_Heart_Training_Optimization

## Heart 데이터로 관찰하는 적합한 학습 조건과 과적합

같은 Heart(심장질환) 데이터와 같은 분류 문제를 사용해도  
전처리, Learning Rate(학습률), 모델 복잡도, L2 Regularization(가중치 규제)에 따라 학습 결과가 달라집니다.

> **핵심 질문**  
> 같은 데이터와 같은 목표를 사용해도 학습 조건에 따라 결과가 어떻게 달라질까?


## 실습 흐름

```text
Heart 기본 이진분류
→ 표준화 적용 기준 모델
→ 표준화 제거
→ 학습률 변경
→ 복잡한 신경망
→ 과적합 관찰
→ L2 규제 적용
→ 검증 성능 비교
→ 테스트 데이터 최종 평가
```

| 실험 | 바꾸는 조건 | 핵심 관찰 |
|---|---|---|
| 1 | 기준 모델 | 비교 기준 성능 |
| 2 | 표준화 제거 | 특성 크기가 학습에 미치는 영향 |
| 3 | 학습률 변경 | 느린 학습·안정적 학습·불안정한 학습 |
| 4 | 모델 복잡도 증가 | 훈련 성능과 검증 성능의 차이 |
| 5 | L2 규제 적용 | 과적합 완화 가능성 |

## 1. 목표

- Standardization(스탠더다이제이션, 표준화)이 학습에 미치는 영향을 비교합니다.
- Learning Rate(러닝 레이트, 학습률)가 너무 작거나 클 때의 손실 변화를 관찰합니다.
- 복잡한 모델의 훈련 손실과 검증 손실을 비교하여 Overfitting(오버피팅, 과적합)을 판단합니다.
- `weight_decay`를 이용한 L2 규제로 과적합 완화 효과를 확인합니다.
- 검증 데이터로 학습 조건을 비교하고, 테스트 데이터는 마지막에만 사용합니다.

**Important sentence**

> Training performance alone does not tell us how well a model generalizes.  
> 훈련 성능만으로는 모델이 새로운 데이터에 얼마나 잘 일반화되는지 알 수 없습니다.

## 2. 준비

### 2.1 라이브러리 불러오기

Colab(코랩)에서는 `heart.csv`를 노트북과 함께 업로드한 뒤 위에서 아래로 실행합니다.

In [ ]:
from pathlib import Path
import copy
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

# 그래프에서 한글 글꼴 경고를 줄이기 위해 제목과 축 이름은 영문으로 표시합니다.
print("PyTorch version:", torch.__version__)
print("Device: CPU")


### 2.2 실험 조건 고정하기

실험마다 데이터 분할이나 초기 가중치가 달라지면 조건을 공정하게 비교하기 어렵습니다.  
따라서 Random Seed(난수 시작값)를 고정합니다.

In [ ]:
RANDOM_SEED = 42

def set_seed(seed=RANDOM_SEED):
    # 실험을 다시 실행해도 가능한 한 같은 결과를 얻도록 난수를 고정합니다.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed()


## 3. Data(데이터)

### 3.1 Heart CSV 불러오기

In [ ]:
data_path = Path("heart.csv")

if not data_path.exists():
    raise FileNotFoundError(
        "heart.csv를 찾을 수 없습니다. Colab 왼쪽 파일 영역에 heart.csv를 업로드하세요."
    )

heart_df = pd.read_csv(data_path, encoding="utf-8-sig")

print("데이터 크기:", heart_df.shape)
print("열 이름:", heart_df.columns.tolist())


### 3.2 CSV 앞 5행과 마지막 5행 확인하기

In [ ]:
print("[앞 5행]")
display(heart_df.head())

print("[마지막 5행]")
display(heart_df.tail())


### 3.3 입력 특성과 목표값 확인하기

- 연속형 특성: `age`, `trestbps`, `chol`, `thalach`, `oldpeak`
- 범주형 특성: 숫자로 저장되어 있지만 값의 크기보다 범주 구분이 중요한 열
- 목표값 `target`: 심장질환 있음 `1`, 없음 `0`

In [ ]:
print("[결측값 개수]")
print(heart_df.isna().sum())

print("\n[target 분포]")
print(heart_df["target"].value_counts().sort_index())

assert heart_df.isna().sum().sum() == 0, "결측값 처리가 필요합니다."
assert set(heart_df["target"].unique()) == {0, 1}, "target은 0과 1이어야 합니다."


## 4. 실습 단계

### 4.1 입력과 정답 분리·범주형 특성 원핫 인코딩

One-Hot Encoding(원핫 인코딩, 범주를 0과 1 열로 변환)을 적용합니다.  
이번 Lab의 목적은 전처리 방법을 새로 배우는 것이 아니므로 `Lab 05_2`의 처리 구조를 그대로 사용합니다.

In [ ]:
target_column = "target"
continuous_columns = ["age", "trestbps", "chol", "thalach", "oldpeak"]
categorical_columns = [
    "sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"
]

X_dataframe = heart_df.drop(columns=[target_column]).copy()
y_array = heart_df[target_column].to_numpy(dtype=np.float32)

X_encoded = pd.get_dummies(
    X_dataframe,
    columns=categorical_columns,
    drop_first=False,
    dtype=np.float32,
)

print("원본 입력 특성 수:", X_dataframe.shape[1])
print("원핫 인코딩 후 입력 특성 수:", X_encoded.shape[1])


### 4.2 훈련·검증·테스트 데이터 분할

```text
Training Set(트레이닝 세트, 훈련 데이터)
→ 가중치 학습

Validation Set(밸리데이션 세트, 검증 데이터)
→ 학습 조건과 과적합 비교

Test Set(테스트 세트, 최종 평가 데이터)
→ 모든 비교가 끝난 뒤 마지막 평가
```

각 집합에 목표값 `0·1`의 비율이 유지되도록 Stratified Split(스트래티파이드 스플릿, 계층적 분할)을 직접 구현합니다.

In [ ]:
def stratified_three_way_indices(labels, train_ratio=0.60, val_ratio=0.20, seed=42):
    # 각 클래스 비율을 유지하며 훈련·검증·테스트 인덱스를 만듭니다.
    rng = np.random.default_rng(seed)
    train_indices, val_indices, test_indices = [], [], []

    for class_value in np.unique(labels):
        class_indices = np.where(labels == class_value)[0]
        rng.shuffle(class_indices)

        train_end = int(len(class_indices) * train_ratio)
        val_end = train_end + int(len(class_indices) * val_ratio)

        train_indices.extend(class_indices[:train_end])
        val_indices.extend(class_indices[train_end:val_end])
        test_indices.extend(class_indices[val_end:])

    for index_list in (train_indices, val_indices, test_indices):
        rng.shuffle(index_list)

    return (
        np.array(train_indices),
        np.array(val_indices),
        np.array(test_indices),
    )


train_indices, val_indices, test_indices = stratified_three_way_indices(
    y_array,
    train_ratio=0.60,
    val_ratio=0.20,
    seed=RANDOM_SEED,
)

print("훈련 데이터:", len(train_indices))
print("검증 데이터:", len(val_indices))
print("테스트 데이터:", len(test_indices))


### 4.3 표준화 데이터와 원본 크기 데이터 준비

표준화는 반드시 **훈련 데이터의 평균과 표준편차**로 계산합니다.

$$
x_{\mathrm{standardized}} = \frac{x-\mu_{\mathrm{train}}}{\sigma_{\mathrm{train}}}
$$

- $x$: 원래 특성값
- $\mu_{\mathrm{train}}$: 훈련 데이터 평균
- $\sigma_{\mathrm{train}}$: 훈련 데이터 표준편차

검증·테스트 데이터에서 평균과 표준편차를 다시 계산하면 미래 데이터의 정보를 미리 사용하는 누출이 발생합니다.

In [ ]:
encoded_columns = X_encoded.columns.tolist()
continuous_indices = [encoded_columns.index(column) for column in continuous_columns]

X_all_raw = X_encoded.to_numpy(dtype=np.float32)

X_train_raw = X_all_raw[train_indices].copy()
X_val_raw = X_all_raw[val_indices].copy()
X_test_raw = X_all_raw[test_indices].copy()

y_train = y_array[train_indices].copy()
y_val = y_array[val_indices].copy()
y_test = y_array[test_indices].copy()

train_mean = X_train_raw[:, continuous_indices].mean(axis=0)
train_std = X_train_raw[:, continuous_indices].std(axis=0)
train_std[train_std == 0] = 1.0

X_train_standardized = X_train_raw.copy()
X_val_standardized = X_val_raw.copy()
X_test_standardized = X_test_raw.copy()

X_train_standardized[:, continuous_indices] = (
    X_train_standardized[:, continuous_indices] - train_mean
) / train_std
X_val_standardized[:, continuous_indices] = (
    X_val_standardized[:, continuous_indices] - train_mean
) / train_std
X_test_standardized[:, continuous_indices] = (
    X_test_standardized[:, continuous_indices] - train_mean
) / train_std

print("표준화 전 연속형 평균:", np.round(X_train_raw[:, continuous_indices].mean(axis=0), 2))
print("표준화 후 연속형 평균:", np.round(X_train_standardized[:, continuous_indices].mean(axis=0), 4))
print("표준화 후 연속형 표준편차:", np.round(X_train_standardized[:, continuous_indices].std(axis=0), 4))


### 4.4 NumPy 배열을 PyTorch 텐서로 변환

In [ ]:
def to_feature_tensor(array):
    return torch.tensor(array, dtype=torch.float32)


def to_target_tensor(array):
    # BCEWithLogitsLoss의 모델 출력 [N, 1]과 같은 형태로 만듭니다.
    return torch.tensor(array, dtype=torch.float32).view(-1, 1)


data_tensors = {
    "standardized": {
        "train_x": to_feature_tensor(X_train_standardized),
        "val_x": to_feature_tensor(X_val_standardized),
        "test_x": to_feature_tensor(X_test_standardized),
    },
    "raw": {
        "train_x": to_feature_tensor(X_train_raw),
        "val_x": to_feature_tensor(X_val_raw),
        "test_x": to_feature_tensor(X_test_raw),
    },
}

train_y_tensor = to_target_tensor(y_train)
val_y_tensor = to_target_tensor(y_val)
test_y_tensor = to_target_tensor(y_test)

input_size = data_tensors["standardized"]["train_x"].shape[1]
print("입력 특성 수:", input_size)
print("훈련 텐서:", data_tensors["standardized"]["train_x"].shape, train_y_tensor.shape)


### 4.5 선형 모델과 복잡한 신경망 정의

`BCEWithLogitsLoss()`는 Sigmoid(시그모이드, 로짓을 확률로 변환)와 BCE(비씨이, 이진 교차엔트로피)를 안정적으로 결합합니다.  
따라서 모델 마지막에 `nn.Sigmoid()`를 추가하지 않고 로짓을 손실함수에 직접 전달합니다.

In [ ]:
class LinearHeartModel(nn.Module):
    # Lab 05_2와 같은 단일 선형층 이진분류 모델입니다.

    def __init__(self, number_of_inputs):
        super().__init__()
        self.output_layer = nn.Linear(number_of_inputs, 1)

    def forward(self, inputs):
        return self.output_layer(inputs)


class ComplexHeartModel(nn.Module):
    # 작은 데이터에 비해 표현력이 큰 다층 신경망입니다.

    def __init__(self, number_of_inputs):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(number_of_inputs, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, inputs):
        return self.network(inputs)


### 4.6 공통 학습 함수 정의

모든 실험은 같은 학습 함수를 사용합니다.  
이 함수는 매 Epoch(에폭, 전체 훈련 데이터 1회 학습)마다 훈련·검증 손실과 정확도를 기록하고, 검증 손실이 가장 낮았던 가중치도 보관합니다.

In [ ]:
def binary_accuracy_from_logits(logits, targets):
    probabilities = torch.sigmoid(logits)
    predictions = (probabilities >= 0.5).float()
    return (predictions == targets).float().mean().item()


def train_experiment(
    model_factory,
    train_x,
    val_x,
    learning_rate,
    number_of_epochs,
    weight_decay=0.0,
    seed=RANDOM_SEED,
):
    set_seed(seed)
    model = model_factory(input_size)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_accuracy": [],
        "val_accuracy": [],
    }

    best_val_loss = float("inf")
    best_epoch = 0
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(number_of_epochs):
        model.train()
        optimizer.zero_grad()

        train_logits = model(train_x)
        train_loss = criterion(train_logits, train_y_tensor)
        train_loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            train_logits = model(train_x)
            val_logits = model(val_x)

            current_train_loss = criterion(train_logits, train_y_tensor).item()
            current_val_loss = criterion(val_logits, val_y_tensor).item()
            current_train_accuracy = binary_accuracy_from_logits(
                train_logits, train_y_tensor
            )
            current_val_accuracy = binary_accuracy_from_logits(
                val_logits, val_y_tensor
            )

        history["train_loss"].append(current_train_loss)
        history["val_loss"].append(current_val_loss)
        history["train_accuracy"].append(current_train_accuracy)
        history["val_accuracy"].append(current_val_accuracy)

        if current_val_loss < best_val_loss:
            best_val_loss = current_val_loss
            best_epoch = epoch + 1
            best_state = copy.deepcopy(model.state_dict())

    return {
        "model": model,
        "best_state": best_state,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "history": history,
        "learning_rate": learning_rate,
        "weight_decay": weight_decay,
    }


### 4.7 결과 평가와 그래프 함수 정의

In [ ]:
def evaluate_model(model, features, targets):
    model.eval()
    criterion = nn.BCEWithLogitsLoss()

    with torch.no_grad():
        logits = model(features)
        loss = criterion(logits, targets).item()
        probabilities = torch.sigmoid(logits)
        predictions = (probabilities >= 0.5).float()
        accuracy = (predictions == targets).float().mean().item()

    return {
        "loss": loss,
        "accuracy": accuracy,
        "probabilities": probabilities.cpu().numpy().ravel(),
        "predictions": predictions.cpu().numpy().astype(int).ravel(),
    }


def plot_loss_curves(results, labels, title, log_scale=False):
    plt.figure(figsize=(10, 5))
    for result, label in zip(results, labels):
        plt.plot(result["history"]["val_loss"], label=label)

    plt.xlabel("Epoch")
    plt.ylabel("Validation Loss")
    plt.title(title)
    if log_scale:
        plt.yscale("log")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()


def plot_train_validation(result, title):
    history = result["history"]
    best_epoch = result["best_epoch"]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    axes[0].plot(history["train_loss"], label="Train Loss")
    axes[0].plot(history["val_loss"], label="Validation Loss")
    axes[0].axvline(best_epoch - 1, color="red", linestyle="--", label="Best Epoch")
    axes[0].set_title(f"{title}: Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    axes[1].plot(history["train_accuracy"], label="Train Accuracy")
    axes[1].plot(history["val_accuracy"], label="Validation Accuracy")
    axes[1].axvline(best_epoch - 1, color="red", linestyle="--", label="Best Epoch")
    axes[1].set_title(f"{title}: Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_ylim(0.0, 1.05)
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.show()


## 5. 실험 1: 기준 모델

표준화된 데이터, 선형 모델, 학습률 `0.01`을 기준 조건으로 사용합니다.

In [ ]:
baseline_result = train_experiment(
    model_factory=LinearHeartModel,
    train_x=data_tensors["standardized"]["train_x"],
    val_x=data_tensors["standardized"]["val_x"],
    learning_rate=0.01,
    number_of_epochs=1000,
)

print("최저 검증 손실 Epoch:", baseline_result["best_epoch"])
print("최저 검증 손실:", round(baseline_result["best_val_loss"], 4))

plot_train_validation(baseline_result, "Baseline Linear Model")


### 실험 1 관찰

- 훈련 손실과 검증 손실이 함께 감소하는 구간은 새로운 데이터에도 유효한 패턴을 학습하는 구간입니다.
- 검증 손실이 가장 낮은 Epoch는 현재 조건에서 일반화가 가장 좋았던 시점의 후보입니다.
- 정확도만 보면 작은 확률 변화가 가려질 수 있으므로 손실 곡선도 함께 봅니다.

## 6. 실험 2: 표준화 제거

모델·학습률·Epoch 수는 유지하고 입력의 표준화만 제거합니다.

> One variable at a time.  
> 한 번에 조건 하나만 바꿔야 결과 변화의 원인을 해석할 수 있습니다.

In [ ]:
raw_scale_result = train_experiment(
    model_factory=LinearHeartModel,
    train_x=data_tensors["raw"]["train_x"],
    val_x=data_tensors["raw"]["val_x"],
    learning_rate=0.01,
    number_of_epochs=1000,
)

plot_loss_curves(
    results=[baseline_result, raw_scale_result],
    labels=["Standardized", "Not Standardized"],
    title="Effect of Standardization",
    log_scale=True,
)

print("표준화 적용 최저 검증 손실:", round(baseline_result["best_val_loss"], 4))
print("표준화 미적용 최저 검증 손실:", round(raw_scale_result["best_val_loss"], 4))


### 실험 2 관찰

- 원본 데이터에서는 `chol`처럼 수백 단위인 특성과 `oldpeak`처럼 한 자릿수인 특성이 섞여 있습니다.
- Adam(아담, 적응형 최적화 알고리즘)은 특성 크기 차이를 일부 보완하지만, 표준화의 필요성이 사라지는 것은 아닙니다.
- 두 곡선의 초기 손실, 감소 속도, 최저 검증 손실을 비교합니다.
- 결과는 실행 환경과 분할에 따라 달라질 수 있으므로 “표준화하면 정확도가 반드시 상승한다”로 단정하지 않습니다.

## 7. 실험 3: 학습률 변경

표준화된 데이터와 선형 모델을 고정하고 학습률만 바꿉니다.

| 학습률 | 예상 관찰 |
|---:|---|
| `0.00001` | 같은 Epoch 동안 매우 느린 학습 |
| `0.01` | 기준 학습 |
| `0.1` | 빠르지만 변동 가능 |
| `1.0` | 최적점을 지나쳐 불안정할 가능성 |

Optimizer(옵티마이저, 가중치 갱신 알고리즘)가 Adam이므로 큰 학습률에서도 손실이 수학적으로 무한대로 발산하지 않을 수 있습니다.  
그 경우에는 손실의 진동, 최저점 이후 악화, 검증 성능 저하를 관찰합니다.

In [ ]:
learning_rate_settings = [0.00001, 0.01, 0.1, 1.0]
learning_rate_results = {}

for learning_rate in learning_rate_settings:
    learning_rate_results[learning_rate] = train_experiment(
        model_factory=LinearHeartModel,
        train_x=data_tensors["standardized"]["train_x"],
        val_x=data_tensors["standardized"]["val_x"],
        learning_rate=learning_rate,
        number_of_epochs=500,
    )

plot_loss_curves(
    results=[learning_rate_results[lr] for lr in learning_rate_settings],
    labels=[f"lr={lr}" for lr in learning_rate_settings],
    title="Validation Loss by Learning Rate",
    log_scale=True,
)

learning_rate_summary = pd.DataFrame(
    [
        {
            "learning_rate": learning_rate,
            "best_epoch": result["best_epoch"],
            "best_validation_loss": result["best_val_loss"],
            "last_validation_loss": result["history"]["val_loss"][-1],
            "last_validation_accuracy": result["history"]["val_accuracy"][-1],
        }
        for learning_rate, result in learning_rate_results.items()
    ]
)

display(learning_rate_summary.round(4))


### 실험 3 관찰

```text
학습률이 너무 작음
→ 한 번의 가중치 이동이 작음
→ 제한된 Epoch에서는 학습이 덜 진행됨

학습률이 적절함
→ 손실이 빠르고 안정적으로 감소

학습률이 너무 큼
→ 좋은 지점을 지나치거나 손실이 크게 변동
→ 마지막 성능이 최저 성능보다 나빠질 수 있음
```

Best Validation Loss(베스트 밸리데이션 로스, 최저 검증 손실)와 Last Validation Loss(라스트 밸리데이션 로스, 마지막 검증 손실)를 함께 비교합니다.

## 8. 실험 4: 복잡한 모델과 과적합

선형층 1개 대신 은닉층 3개를 가진 모델을 충분히 오래 학습합니다.

```text
입력
→ Linear(input_size, 128)
→ ReLU
→ Linear(128, 64)
→ ReLU
→ Linear(64, 32)
→ ReLU
→ Linear(32, 1)
```

훈련 손실은 계속 감소하는데 검증 손실이 어느 시점부터 증가한다면 과적합 신호입니다.

In [ ]:
complex_result = train_experiment(
    model_factory=ComplexHeartModel,
    train_x=data_tensors["standardized"]["train_x"],
    val_x=data_tensors["standardized"]["val_x"],
    learning_rate=0.01,
    number_of_epochs=2000,
    weight_decay=0.0,
)

print("최저 검증 손실 Epoch:", complex_result["best_epoch"])
print("최저 검증 손실:", round(complex_result["best_val_loss"], 4))
print("마지막 검증 손실:", round(complex_result["history"]["val_loss"][-1], 4))

plot_train_validation(complex_result, "Complex Neural Network")


### 과적합 판단 기준

| 훈련 데이터 | 검증 데이터 | 판단 |
|---|---|---|
| 손실 감소 | 손실 감소 | 유효한 패턴 학습 |
| 손실 감소 | 손실 정체 | 일반화 개선이 멈춤 |
| 손실 계속 감소 | 손실 다시 증가 | 과적합 가능성이 큼 |
| 정확도 매우 높음 | 정확도 정체·하락 | 훈련 데이터 암기 가능성 |

과적합은 단순히 “훈련 정확도가 높다”는 뜻이 아닙니다.  
**훈련 성능과 검증 성능의 간격이 커지는 현상**을 함께 확인해야 합니다.

## 9. 실험 5: L2 규제 적용

모델과 학습률은 유지하고 Adam의 `weight_decay`만 `0.001`로 설정합니다.

L2 규제는 가중치가 지나치게 커지는 것을 억제합니다.

$$
\mathrm{Loss}_{\mathrm{total}}
=
\mathrm{Loss}_{\mathrm{data}}
+
\lambda \sum_i w_i^2
$$

- $\mathrm{Loss}_{\mathrm{data}}$: 원래 분류 손실
- $w_i$: 각 가중치
- $\lambda$: 규제 강도

PyTorch의 Adam에서 `weight_decay`는 실습 수준에서 L2 규제 효과를 관찰하는 간단한 방법입니다.

In [ ]:
regularized_result = train_experiment(
    model_factory=ComplexHeartModel,
    train_x=data_tensors["standardized"]["train_x"],
    val_x=data_tensors["standardized"]["val_x"],
    learning_rate=0.01,
    number_of_epochs=2000,
    weight_decay=0.001,
)

plot_loss_curves(
    results=[complex_result, regularized_result],
    labels=["Complex Model", "Complex Model + L2"],
    title="Effect of L2 Regularization",
    log_scale=True,
)

plot_train_validation(regularized_result, "Complex Neural Network + L2")


### 실험 5 관찰

- L2 규제를 적용하면 훈련 정확도가 약간 낮아질 수 있습니다.
- 중요한 것은 훈련 점수의 최고값이 아니라 검증 손실과 일반화 성능입니다.
- 규제가 너무 약하면 차이가 작고, 너무 강하면 Underfitting(언더피팅, 과소적합)이 생길 수 있습니다.
- 데이터가 작으므로 한 번의 결과보다 여러 규제 강도를 비교하는 것이 더 안전합니다.

## 10. 결과 확인

### 10.1 검증 데이터 기준 전체 실험 비교

아직 테스트 데이터는 사용하지 않습니다.  
각 실험의 **검증 손실이 가장 낮았던 가중치**를 복원하여 공정하게 비교합니다.

In [ ]:
experiment_results = {
    "Baseline: standardized linear": {
        "result": baseline_result,
        "model_factory": LinearHeartModel,
        "data_key": "standardized",
    },
    "No standardization: linear": {
        "result": raw_scale_result,
        "model_factory": LinearHeartModel,
        "data_key": "raw",
    },
    "Complex model": {
        "result": complex_result,
        "model_factory": ComplexHeartModel,
        "data_key": "standardized",
    },
    "Complex model + L2": {
        "result": regularized_result,
        "model_factory": ComplexHeartModel,
        "data_key": "standardized",
    },
}

validation_rows = []

for experiment_name, experiment in experiment_results.items():
    restored_model = experiment["model_factory"](input_size)
    restored_model.load_state_dict(experiment["result"]["best_state"])

    data_key = experiment["data_key"]
    validation_metrics = evaluate_model(
        restored_model,
        data_tensors[data_key]["val_x"],
        val_y_tensor,
    )

    validation_rows.append(
        {
            "experiment": experiment_name,
            "best_epoch": experiment["result"]["best_epoch"],
            "learning_rate": experiment["result"]["learning_rate"],
            "weight_decay": experiment["result"]["weight_decay"],
            "validation_loss": validation_metrics["loss"],
            "validation_accuracy": validation_metrics["accuracy"],
        }
    )

validation_summary = pd.DataFrame(validation_rows).sort_values("validation_loss")
display(validation_summary.round(4))


### 10.2 테스트 데이터 최종 평가

이제 처음으로 테스트 데이터를 사용합니다.  
검증 결과표는 학습 조건 비교용이고, 아래 테스트 결과표는 보지 않았던 데이터에 대한 최종 일반화 확인용입니다.

> The test set is for final evaluation, not repeated tuning.  
> 테스트 데이터는 반복적인 조건 조정이 아니라 최종 평가에 사용합니다.

In [ ]:
test_rows = []
restored_models = {}

for experiment_name, experiment in experiment_results.items():
    restored_model = experiment["model_factory"](input_size)
    restored_model.load_state_dict(experiment["result"]["best_state"])
    restored_models[experiment_name] = restored_model

    data_key = experiment["data_key"]
    train_metrics = evaluate_model(
        restored_model,
        data_tensors[data_key]["train_x"],
        train_y_tensor,
    )
    test_metrics = evaluate_model(
        restored_model,
        data_tensors[data_key]["test_x"],
        test_y_tensor,
    )

    test_rows.append(
        {
            "experiment": experiment_name,
            "train_accuracy": train_metrics["accuracy"],
            "test_loss": test_metrics["loss"],
            "test_accuracy": test_metrics["accuracy"],
            "train_test_accuracy_gap": (
                train_metrics["accuracy"] - test_metrics["accuracy"]
            ),
        }
    )

test_summary = pd.DataFrame(test_rows).sort_values("test_loss")
display(test_summary.round(4))


### 10.3 혼동행렬과 이진분류 평가지표

Accuracy(애큐러시, 정확도)만으로는 어떤 클래스를 틀렸는지 알 수 없습니다.  
기준 모델, 복잡한 모델, L2 규제 모델을 Confusion Matrix(컨퓨전 매트릭스, 혼동행렬)와 함께 비교합니다.

In [ ]:
def calculate_binary_metrics(targets, predictions):
    targets = np.asarray(targets, dtype=int)
    predictions = np.asarray(predictions, dtype=int)

    tn = int(np.sum((targets == 0) & (predictions == 0)))
    fp = int(np.sum((targets == 0) & (predictions == 1)))
    fn = int(np.sum((targets == 1) & (predictions == 0)))
    tp = int(np.sum((targets == 1) & (predictions == 1)))

    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1_score = 2 * precision * recall / max(precision + recall, 1e-12)

    return {
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score,
    }


models_to_compare = [
    "Baseline: standardized linear",
    "Complex model",
    "Complex model + L2",
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metric_rows = []

for axis, experiment_name in zip(axes, models_to_compare):
    model = restored_models[experiment_name]
    test_result = evaluate_model(
        model,
        data_tensors["standardized"]["test_x"],
        test_y_tensor,
    )
    metrics = calculate_binary_metrics(y_test, test_result["predictions"])
    metric_rows.append({"experiment": experiment_name, **metrics})

    confusion_matrix = np.array(
        [[metrics["tn"], metrics["fp"]], [metrics["fn"], metrics["tp"]]]
    )
    image = axis.imshow(confusion_matrix, cmap="Blues")
    axis.set_title(experiment_name)
    axis.set_xlabel("Predicted")
    axis.set_ylabel("Actual")
    axis.set_xticks([0, 1])
    axis.set_yticks([0, 1])

    for row in range(2):
        for column in range(2):
            axis.text(
                column,
                row,
                confusion_matrix[row, column],
                ha="center",
                va="center",
            )

plt.tight_layout()
plt.show()

metric_summary = pd.DataFrame(metric_rows)
display(
    metric_summary[
        ["experiment", "accuracy", "precision", "recall", "f1_score"]
    ].round(4)
)


### 10.4 실행 검증

다음 조건이 모두 통과하면 데이터 분할, 표준화, 학습 기록, 확률 계산의 기본 구조가 정상입니다.

In [ ]:
assert len(set(train_indices) & set(val_indices)) == 0
assert len(set(train_indices) & set(test_indices)) == 0
assert len(set(val_indices) & set(test_indices)) == 0

all_indices = np.concatenate([train_indices, val_indices, test_indices])
assert len(np.unique(all_indices)) == len(heart_df)

assert np.allclose(
    X_train_standardized[:, continuous_indices].mean(axis=0),
    0.0,
    atol=1e-5,
)
assert np.allclose(
    X_train_standardized[:, continuous_indices].std(axis=0),
    1.0,
    atol=1e-5,
)

assert len(baseline_result["history"]["train_loss"]) == 1000
assert len(complex_result["history"]["val_loss"]) == 2000

for model in restored_models.values():
    sample_result = evaluate_model(
        model,
        data_tensors["standardized"]["test_x"][:5],
        test_y_tensor[:5],
    )
    assert np.all((sample_result["probabilities"] >= 0.0))
    assert np.all((sample_result["probabilities"] <= 1.0))

print("모든 실행 검증을 통과했습니다.")


## 11. 핵심 정리

| 개념 | 관찰 기준 | 핵심 의미 |
|---|---|---|
| 표준화 | 초기 손실·감소 속도·최저 검증 손실 | 특성 크기를 비슷하게 맞춰 학습을 안정화 |
| 학습률 | 최저 손실·마지막 손실·곡선 변동 | 가중치를 얼마나 크게 이동할지 결정 |
| 과적합 | 훈련 손실 감소 + 검증 손실 증가 | 훈련 데이터에 지나치게 맞춰짐 |
| L2 규제 | 훈련·검증 간격과 검증 손실 변화 | 큰 가중치를 억제해 복잡도를 완화 |
| 검증 데이터 | 조건 비교 | 학습률·모델·규제 선택에 사용 |
| 테스트 데이터 | 최종 평가 | 반복적인 조건 선택에 사용하지 않음 |
